<a href="https://colab.research.google.com/github/Md-Salman-Rahman339/AI-ML/blob/main/Copy_of_DL_Assignment_03_Question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# DL Assignment 03

**Name:**Md.Salman Rohoman Nayeem

**Course Email:**  mnayeem2310484@bscse.uiu.ac.bd


## End of Assignment

Before submitting:
- Run all cells from top to bottom.  
- Check that all answer sections are filled.  
- Instruction video অনুযায়ী আমাদের দেয়া Colab ফাইলটি থেকে প্রথম একটি Save copy in drive করে নিবা। এরপর Google colab এর মধ্যে কোডগুলো করবে এবং সেই ফাইলটি ‘Anyone with the link’ & ‘View’ Access দিয়ে ফাইলটির Shareble Link টি সাবমিট করবে।

# General Instruction

You must choose your own dataset.

The dataset must:

Be a supervised learning dataset (Regression or Binary Classification)

Contain at least 300 samples

Have at least 2 input features

Be in CSV format

You are NOT allowed to use Dataset or DataLoader.

You must implement everything manually.

# Question 01: [ Marks 05 ]

## Dataset Preparation

## Using your chosen dataset:

Load the dataset.

Perform necessary preprocessing:

Handle missing values (if any)

Encode categorical variables (if necessary)

Feature scaling (if needed)

Separate features (X) and target (y).

Convert them into NumPy arrays.

Convert them into PyTorch tensors.

Split into training and testing sets.

Clearly explain each preprocessing decision.

# **Write** Answer 01:


In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

df =pd.read_csv('/content/insurance.csv')
df.head()


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [6]:
# handle missing valiue
print(df.isnull().sum())

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64


In [7]:
from sklearn.preprocessing import LabelEncoder
# encode categorical variables
cat_features=df.select_dtypes(include='object').columns
print("categorical columns:",list(cat_features))


le=LabelEncoder()
for col in cat_features:
  df[col]=le.fit_transform(df[col])
df.head()

categorical columns: ['sex', 'smoker', 'region']


,age,sex,bmi,children,smoker,region,charges
0,19,0,27.900,0,1,3,16884.92400
1,18,1,33.770,1,0,2,1725.55230
2,28,1,33.000,3,0,2,4449.46200
3,33,1,22.705,0,0,1,21984.47061
4,32,1,28.880,0,0,1,3866.85520


In [32]:
# feature scaling
from sklearn.preprocessing import StandardScaler
scaler_x=StandardScaler()

numerical_cols=['age','bmi','children']
df[numerical_cols]=scaler_x.fit_transform(df[numerical_cols])


In [10]:
X=df.drop('charges',axis=1)
y=df['charges']


In [11]:
# Train test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)




In [35]:
scaler_y = StandardScaler()

y_train = scaler_y.fit_transform(y_train.reshape(-1,1))
y_test = scaler_y.transform(y_test.reshape(-1,1))

In [37]:
# numpy to tensor
X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

print(X_train_tensor.shape)
print(y_train_tensor.shape)

torch.Size([1070, 6])
torch.Size([1070, 1])


Insurance dataset->Regression problem
target: charges

# Question 02: [ Marks 20 ]

## Design a neural network using nn.Module.

### The model must contain:

Input layer

At least one hidden layer

Output layer

Suitable activation function



## Justify:

Number of hidden neurons

Choice of activation function

Print  the total number of trainable parameters.


## Write Answer 02:


In [38]:
class InsuranceModel(nn.Module):
  def __init__(self):
    super(InsuranceModel,self).__init__()

    self.model=nn.Sequential(
        nn.Linear(6,16),
        nn.ReLU(),
        nn.Linear(16,8),
        nn.ReLU(),
        nn.Linear(8,1),

    )

  def forward(self,x):
    return self.model(x)
model=InsuranceModel()
print(model)



InsuranceModel(
  (model): Sequential(
    (0): Linear(in_features=6, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=8, bias=True)
    (3): ReLU()
    (4): Linear(in_features=8, out_features=1, bias=True)
  )
)


First hidden layer:16 neurons
second hidden layer:8 neurons so total hidden neurons is 24 .

Activation Function:ReLU cause it avoids vanishin gradient problem .Computationally efficient and it works well for regression tasks

In [39]:
#  total number of trainable parameters.
total=0
for p in model.parameters():
  if p.requires_grad:
    total+=p.numel()
print(total)

257


# Question 03: [ Marks 10 ]

Choose an appropriate loss function.

Choose an optimizer.

<br>

Justify your choices based on:

Regression vs Classification

Nature of the dataset

## Write Answer 03:

In [40]:
loss_fn=nn.MSELoss()
optimizer=torch.optim.SGD(model.parameters(),lr=0.01)
print(loss_fn)
print(optimizer)


MSELoss()
SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    lr: 0.01
    maximize: False
    momentum: 0
    nesterov: False
    weight_decay: 0
)


# Regression vs Classification:
Target=charges->continuous value
so this is regression problem
that's why use MSELoss->best for regression

# Nature of Dataset:
Medium size(1300) samples,Numerical and encoded categorical features

# Question 04: [ Marks 15 ]

## Implement a full training loop:

Forward pass

Loss computation

Backward pass

Parameter update

Gradient reset

### Requirements:

Train for at least 100 epochs.

Print loss every 10 epochs.

Store training loss history(You can pick your own Data Structure).

Explain clearly what happens in each step of the pipeline.

## Write Answer 04:

In [41]:
epochs=100
loss_history=[]
for epoch in range(epochs):
  optimizer.zero_grad()
  y_pred=model(X_train_tensor)

  loss=loss_fn(y_pred,y_train_tensor.view(-1,1))

  loss.backward()
  optimizer.step()

  loss_history.append(loss.item())
  if(epoch+1)%10==0:
    print(f"Epoch {epoch+1}, Loss:{loss.item()}")


Epoch 10, Loss:1.0011013746261597
Epoch 20, Loss:0.9989567995071411
Epoch 30, Loss:0.9968867301940918
Epoch 40, Loss:0.9948367476463318
Epoch 50, Loss:0.9927715063095093
Epoch 60, Loss:0.9906959533691406
Epoch 70, Loss:0.9885767102241516
Epoch 80, Loss:0.9863776564598083
Epoch 90, Loss:0.9840875267982483
Epoch 100, Loss:0.9817011952400208


Forward pass compute predictions ,loss function calculates error between predicted and actual values ,backward pass computes gradients using backpropagation,optimizer updates model weights  ,gradients are reset to avoid accumulation and this process repeats for multiple epochs to minimize loss

# Question 05: [ Marks 10 ]

## Evaluate the model on test data.

## For regression:

Report MSE and MAE


## For classification:

Report Accuracy

Compare training vs testing performance.

State whether the model is underfitting or overfitting.

## Write Answer 05:

In [42]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

model.eval()

with torch.no_grad():
  y_test_pred=model(X_test_tensor)
  y_test_pred=y_test_pred.numpy()

  y_test_actual=y_test_tensor.numpy()

  mse=mean_squared_error(y_test_actual,y_test_pred)
  mae=mean_absolute_error(y_test_actual,y_test_pred)

  print(f"MSE:{mse}")
  print(f"MAE:{mae}")


MSE:1.0615555047988892
MAE:0.7893550992012024


# Question 06: [ Marks 20 ]

## Modify at least ONE of the following:

Learning rate

Number of hidden neurons

Number of epochs

### Train again and compare:

Convergence speed

Final performance

Explain how the change affected the model.

## Write Answer 06:

# Question 07: [ Marks 20 ]


# Training Analysis

Answer the following:

Why must gradients be reset every epoch?

What happens if learning rate is too high?

What happens if learning rate is too small?

Why do we define layers inside the constructor (__init__) and not inside forward()?


## Write Answer 07: